In [31]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(r"E:/Portfolio/RAG/data/document.txt",encoding="utf-8")
textfile = loader.load()
print(textfile)


[Document(metadata={'source': 'E:/Portfolio/RAG/data/document.txt'}, page_content='I’ve been a data scientist for 6 years now. If you’d asked me what I’m an expert in, I would have confidently said “classification models for tabular data”.\n\nBut here’s the thing about thinking you’re an expert: that’s usually when you discover how much more there is to learn.\n\nA few weeks ago, while working on a project that depended on probability prediction, I had one of those “Wait, have I been doing this wrong the whole time?” moments.\n\nBut let’s start at the beginning.\n\nStep one: What really matters?\nTo choose the right metric, you first have to figure out what really matters for your problem.\n\nIn my case, I was working on a model whose outputs directly influenced economic value calculations. So it wasn’t just about getting the classes right, it was about getting the probabilities right.\n\nA model that is able to estimate good probabilities is called a calibrated model.\n\nIn practice, 

In [32]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader(r"E:/Portfolio/RAG/data/Advance Assignment.pdf")
documents=loader.load()

RAG Pipelines - Data Ingestion to Vector DB pipeline

In [36]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunkDocs= text_splitter.split_documents(documents)
chunkDocs[:5]

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'TeX', 'creationdate': '2023-12-05T06:40:02+00:00', 'moddate': '2023-12-05T06:40:02+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'trapped': '/False', 'source': 'E:/Portfolio/RAG/data/Advance Assignment.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content='Advanced Algorithms and Applications\nAssignment 5\n1.Solution to Red-Black Tree Violation\nIdentification of Violation\nUpon inserting a new node into a red-black tree, we encounter a violation of\nthe properties of red-black trees. Specifically, a newly inserted node z creates a\nsituation where a red node has a red child, which is not allowed.\nCase Analysis\n• In the first tree, node z is red and its parent node is also red, which is a\ndirect violation of the red-black tree property that red nodes cannot have\nred children (Case 1).\n• The second tree presents the same issue with node

In [49]:
print(len(chunkDocs))

13


In [44]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [61]:
#vector Embeddings
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
embeddings = OpenAIEmbeddings()
db=Chroma.from_documents(documents[:20],embeddings)


In [62]:
ids=db.add_documents(documents=chunkDocs)

In [63]:
ids

['f064d59b-30fa-43c6-989c-8cc8c8070662',
 '663ceca1-ac17-4052-b9e6-6da8f32c9920',
 '11354f73-4ceb-40af-9a1e-67b829b781ef',
 'd50ec1ee-12f6-4987-acfb-b9ed45be4f4f',
 '5a10a779-0a52-46d0-81ef-fc7d69f73641',
 '1fd7508d-40a8-401b-b388-86f0f2ab5bb4',
 'c0243fb0-3e5f-40be-9515-a423e0854241',
 'bb80f38e-449d-49d5-a5c9-69c98721078d',
 '9b63d8aa-d9b4-4d95-bc20-3f147daede92',
 '6c765e84-b6ba-4f59-890a-4fc4a0b75c30',
 '51a614cb-b9ac-4f3f-8af2-988a868d6d87',
 '8f87644e-110b-4be1-918d-a417049ae642',
 '64cf0b5e-a1ed-496b-9546-35fa6ce6b1fe']

In [64]:
results=db.similarity_search(
    "red-black trees"
)
print(results[0])

page_content='Advanced Algorithms and Applications
Assignment 5
1.Solution to Red-Black Tree Violation
Identification of Violation
Upon inserting a new node into a red-black tree, we encounter a violation of
the properties of red-black trees. Specifically, a newly inserted node z creates a
situation where a red node has a red child, which is not allowed.
Case Analysis
• In the first tree, node z is red and its parent node is also red, which is a
direct violation of the red-black tree property that red nodes cannot have
red children (Case 1).
• The second tree presents the same issue with node z being red and having
a red parent (Case 1).
Fixing the Violations
To fix the red-black tree after the insertion of z, we must follow these steps:
First Tree
1. Recolor the parent of z and its sibling to black.
2. Recolor the grandparent of z to red.
3. Since the grandparent is now red and its parent is black, no further action
is necessary as the properties are restored.
Second Tree' metadata={'

In [65]:
results

[Document(metadata={'creator': 'TeX', 'total_pages': 8, 'creationdate': '2023-12-05T06:40:02+00:00', 'producer': 'pdfTeX-1.40.25', 'source': 'E:/Portfolio/RAG/data/Advance Assignment.pdf', 'moddate': '2023-12-05T06:40:02+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'page': 0, 'page_label': '1'}, page_content='Advanced Algorithms and Applications\nAssignment 5\n1.Solution to Red-Black Tree Violation\nIdentification of Violation\nUpon inserting a new node into a red-black tree, we encounter a violation of\nthe properties of red-black trees. Specifically, a newly inserted node z creates a\nsituation where a red node has a red child, which is not allowed.\nCase Analysis\n• In the first tree, node z is red and its parent node is also red, which is a\ndirect violation of the red-black tree property that red nodes cannot have\nred children (Case 1).\n• The second tree presents the same issue with node

In [66]:
retriever= db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5},
)

retriever.batch(
    [
        "What is red-black trees"
        "The process involves adding the least weight edge at each step"
    ],
)

[[Document(metadata={'source': 'E:/Portfolio/RAG/data/Advance Assignment.pdf', 'total_pages': 8, 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creationdate': '2023-12-05T06:40:02+00:00', 'page': 0, 'producer': 'pdfTeX-1.40.25', 'moddate': '2023-12-05T06:40:02+00:00', 'creator': 'TeX', 'page_label': '1'}, page_content='Advanced Algorithms and Applications\nAssignment 5\n1.Solution to Red-Black Tree Violation\nIdentification of Violation\nUpon inserting a new node into a red-black tree, we encounter a violation of\nthe properties of red-black trees. Specifically, a newly inserted node z creates a\nsituation where a red node has a red child, which is not allowed.\nCase Analysis\n• In the first tree, node z is red and its parent node is also red, which is a\ndirect violation of the red-black tree property that red nodes cannot have\nred children (Case 1).\n• The second tree presents the same issue with nod

In [70]:
#Query
query="what is red black tree?"
query_embed=embeddings.embed_query(query)

In [71]:
print(query_embed)

[-0.023002735578759124, -0.03513826608718655, -0.013807406420550335, -0.012928229483454798, -0.02824897462882591, -0.013792994202974494, -0.022080320126290815, -0.02638973057499083, 0.0022952292351582364, -0.03245749489788192, -0.001705207483103888, 0.00775405524655771, -0.015738713424909876, -0.0065505913068116305, -0.0035797648091516238, 0.02791748151738745, 0.03257279636377915, 0.009454758387800173, 0.016805256641072342, -0.010067300452177717, -0.01817446667087634, 0.034331152100615464, -0.016243160597839423, 0.0055957472006443985, 0.016444939094482195, 0.00592723984642155, 0.029747899273425603, -0.001563782534074177, 0.011595050463251716, 0.008265707131291246, 0.02094171302828127, -0.02431429487561451, 0.0015565760760402722, -0.0072063709552393235, -0.021402920754515425, -0.0057650965779269345, -0.006734353367331321, -0.013720930321127416, 0.006644273980683783, 0.016848494225122492, 0.026966239767122212, -0.011883305990640029, -0.011926543574690179, -0.007044227686744708, -0.016444

Retrieve top matching chunks

In [72]:
results=db.similarity_search(
    query,k=5,
)
results

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'TeX', 'page_label': '1', 'source': 'E:/Portfolio/RAG/data/Advance Assignment.pdf', 'trapped': '/False', 'page': 0, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creationdate': '2023-12-05T06:40:02+00:00', 'total_pages': 8, 'moddate': '2023-12-05T06:40:02+00:00'}, page_content='Advanced Algorithms and Applications\nAssignment 5\n1.Solution to Red-Black Tree Violation\nIdentification of Violation\nUpon inserting a new node into a red-black tree, we encounter a violation of\nthe properties of red-black trees. Specifically, a newly inserted node z creates a\nsituation where a red node has a red child, which is not allowed.\nCase Analysis\n• In the first tree, node z is red and its parent node is also red, which is a\ndirect violation of the red-black tree property that red nodes cannot have\nred children (Case 1).\n• The second tree presents the same issue with node

In [73]:
def get_page_number(doc):
    return doc.metadata['page']
sorted_documents=sorted(results,key=get_page_number)

In [74]:
sorted_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'TeX', 'page_label': '1', 'source': 'E:/Portfolio/RAG/data/Advance Assignment.pdf', 'trapped': '/False', 'page': 0, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creationdate': '2023-12-05T06:40:02+00:00', 'total_pages': 8, 'moddate': '2023-12-05T06:40:02+00:00'}, page_content='Advanced Algorithms and Applications\nAssignment 5\n1.Solution to Red-Black Tree Violation\nIdentification of Violation\nUpon inserting a new node into a red-black tree, we encounter a violation of\nthe properties of red-black trees. Specifically, a newly inserted node z creates a\nsituation where a red node has a red child, which is not allowed.\nCase Analysis\n• In the first tree, node z is red and its parent node is also red, which is a\ndirect violation of the red-black tree property that red nodes cannot have\nred children (Case 1).\n• The second tree presents the same issue with node

In [75]:
combined_content="\n\n".join([doc.page_content for doc in sorted_documents])

In [76]:
combined_content

'Advanced Algorithms and Applications\nAssignment 5\n1.Solution to Red-Black Tree Violation\nIdentification of Violation\nUpon inserting a new node into a red-black tree, we encounter a violation of\nthe properties of red-black trees. Specifically, a newly inserted node z creates a\nsituation where a red node has a red child, which is not allowed.\nCase Analysis\n• In the first tree, node z is red and its parent node is also red, which is a\ndirect violation of the red-black tree property that red nodes cannot have\nred children (Case 1).\n• The second tree presents the same issue with node z being red and having\na red parent (Case 1).\nFixing the Violations\nTo fix the red-black tree after the insertion of z, we must follow these steps:\nFirst Tree\n1. Recolor the parent of z and its sibling to black.\n2. Recolor the grandparent of z to red.\n3. Since the grandparent is now red and its parent is black, no further action\nis necessary as the properties are restored.\nSecond Tree\n\nAd

In [93]:
#LLM
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-3.5-turbo")
output_parser=StrOutputParser()
"""from langchain_community.llms import Ollama
##load Ollama LLAM3.2 LLM model
llm=Ollama(model="llama3.2")
llm"""

'from langchain_community.llms import Ollama\n##load Ollama LLAM3.2 LLM model\nllm=Ollama(model="llama3.2")\nllm'

In [94]:
## design ChatPrompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context.
Think step by step before providing a detail answer. I will tip you $1000 if the user finds answer helpful.                                    
<context>
{combined_content}
<context>
Question:{query}
Answer:                                        
""")

In [95]:
print(prompt)

input_variables=['combined_content', 'query'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['combined_content', 'query'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context.\nThink step by step before providing a detail answer. I will tip you $1000 if the user finds answer helpful.                                    \n<context>\n{combined_content}\n<context>\nQuestion:{query}\nAnswer:                                        \n'), additional_kwargs={})]


In [96]:
from langchain_core.output_parsers import StrOutputParser
rag_chain = prompt | llm | StrOutputParser()

response = rag_chain.invoke({
    "combined_content": combined_content,
    "query": query
})

print(response)

A red-black tree is a type of self-balancing binary search tree where each node has an extra bit, called its color, which is used to ensure the tree remains balanced during insertions and deletions. The color of a node can either be red or black, and the tree must adhere to certain properties such as every node being either red or black, the root being black, no red node having a red child, and every path from a node to a null link encountering the same number of black nodes.
